In [ ]:


##import file
import pandas as pd
import numpy as np
df = pd.read_csv("upgraded_financial_dataset.csv")
print(df.head())

In [ ]:
##convert to datetime
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df

In [ ]:
##create columns to isolate date, months and days from each other
df['hour'] = df['TransactionDate'].dt.hour
df['day'] = df['TransactionDate'].dt.day
df['month'] = df['TransactionDate'].dt.month
df['day_of_week'] = df['TransactionDate'].dt.day_name()
df

In [ ]:
##calculate mean, standard deviation, count and sum of amount grouped by customer ID
customer_stats = df.groupby('CustomerID').agg({'Amount':['mean', 'std', 'count'], 'IsFraud': 'sum',}).reset_index()
customer_stats.columns = ['CustomerID', 'avg_amount', 'std_amount', 'transaction_count', 'fraud_count']
print(customer_stats.head())

In [ ]:
##merge customer_stats to main dataframe
df = df.merge(customer_stats, on='CustomerID', how='left')
df

In [ ]:
##calculation of anomally
df['A_score'] = (df['Amount'] - df['avg_amount']) / df['std_amount']
df['A_score']

In [ ]:
##check transactions that fall between odd hours and group them as potential risks
def check_night(x):
    if x >= 22 or x <= 5:
        return 1
    else:
        return 0
df['odd_hours'] = df['hour'].apply(check_night)
df

In [ ]:
##check transactions that falls at odd hours
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
df

In [ ]:
##transaction frequency
freq = df.groupby('CustomerID')['TransactionID'].count().reset_index()
freq.columns = ['CustomerID', 'transaction_freq']

df = df.merge(freq, on='CustomerID', how='left')
df

In [ ]:
##calculation of risk score
df['risk_score'] = (
    df['A_score'] * 0.5 + df['is_night'] * 0.2 + (df['transaction_count'] / df['transaction_count'].max()) * 0.3)

In [ ]:
df['risk_score']

In [ ]:
analysis = df[['CustomerID', 'Amount', 'A_score', 'risk_score', 'avg_amount', 'std_amount', 'fraud_count', 'is_night', 'hour', 'day_of_week']]
print(analysis)

In [ ]:
##check if any duplicate
df['CustomerID'].duplicated().any()

In [ ]:
#because there are duplicates so i want to group the risk analysis by customer id
risk_analysis = analysis.groupby('CustomerID').agg({'Amount': 'mean', 'A_score': 'mean','risk_score': 'mean', 'avg_amount': 'first', 'std_amount': 'first', 'fraud_count': 'first', 'is_night': 'sum', 'hour': 'mean', 'day_of_week': 'nunique'}).reset_index()
risk_analysis.columns = ['CustomerID', 'Amount', 'A_score', 'risk_score', 'avg_amount', 'std_amount', 'fraud_count', 'is_night', 'hour', 'day_of_week']
print(risk_analysis)

In [ ]:
## export tem
risk_analysis.to_csv('1st_risk_analysis.csv', index=False)

In [ ]:
##group into high and low
def rule(x):
    if x > 2:
        return 'High'
    else:
        return 'Low'

df['Rule_based'] =df['A_score'].apply(rule)
df['Rule_based']

In [ ]:
##export data out to csv
df.to_csv("1st_fraud_analysis_project.csv", index=False)